In [14]:
# Environment check: confirm the notebook runs on paper2_env (Python 3.9),
# not the system Python. Should print a path containing "paper2_env".
import sys
print(sys.executable)

/opt/miniconda3/envs/paper2_env/bin/python3.9


In [68]:
# Config check: confirm config.py loads and its settings are readable.
# If this errors with "no attribute", restart the kernel (Kernel > Restart).
import importlib, config
importlib.reload(config)
print(config.DATASETS["uavcan"])
print(config.SEED, config.TOP_K)

/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/processed_UAVCAN.csv
42 5


In [20]:
# Inspect raw UAVCAN data: shape, columns, label values, missing values

# how many rows/columns, the column names, 
# what the label values look like (numbers? text?), 
# and whether there are missing values to clean. 

import pandas as pd, config
df = pd.read_csv(config.DATASETS["uavcan"])
print("shape:", df.shape)
print("columns:", list(df.columns))
print("label values:", df["label"].unique())
print("missing values:", df.isna().sum().sum())

shape: (2032530, 11)
columns: ['CAN_ID', 'DLC', 'byte0', 'byte1', 'byte2', 'byte3', 'byte4', 'byte5', 'byte6', 'byte7', 'label']
label values: [1 0]
missing values: 0


In [71]:
# Test data_loader on UAVCAN: load, clean, scale, split.
# Confirms train/test shapes (should be 70/30),
# feature names, and class balance in the training set.

import numpy as np
import importlib, data_loader
importlib.reload(config)  
importlib.reload(data_loader)

X_train, X_test, y_train, y_test, feat = data_loader.load_dataset("uavcan")
print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print("features:", feat)
print("label balance:", dict(zip(*np.unique(y_train, return_counts=True))))

train shape: (1422771, 10)
test shape: (609759, 10)
features: ['CAN_ID', 'DLC', 'byte0', 'byte1', 'byte2', 'byte3', 'byte4', 'byte5', 'byte6', 'byte7']
label balance: {0: 529561, 1: 893210}


In [72]:
# Test data_loader on all 3 datasets at once.
# Confirms each loads cleanly and shows shape, feature count, and class set.
# Note the differences: uavcan=binary, isot=10 classes, uav_attack=3 classes.

for name in ["uavcan", "isot", "uav_attack"]:
    Xtr, Xte, ytr, yte, feat = data_loader.load_dataset(name)
    print(f"{name:12} train={Xtr.shape}  test={Xte.shape}  n_features={len(feat)}  classes={set(ytr)}")

uavcan       train=(1422771, 10)  test=(609759, 10)  n_features=10  classes={0, 1}
isot         train=(205671, 61)  test=(88145, 61)  n_features=61  classes={0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
uav_attack   train=(7046, 80)  test=(3021, 80)  n_features=80  classes={0, 1, 2}


In [28]:
# Verify benign labels match the CSVs: print label counts per dataset
# and the benign label set in config. Confirms benign points to a real,
# majority-normal class (uavcan=1, isot=0, uav_attack=2).

import importlib, config, data_loader
importlib.reload(config); importlib.reload(data_loader)
import pandas as pd

for name in ["uavcan", "isot", "uav_attack"]:
    df = pd.read_csv(config.DATASETS[name])
    print(name, "label counts:", df["label"].value_counts().to_dict())
    print("   benign set to:", config.BENIGN_LABEL[name])

uavcan label counts: {1: 1276014, 0: 756516}
   benign set to: 1
isot label counts: {0: 128069, 1: 121244, 6: 23122, 4: 10514, 9: 7723, 3: 973, 5: 610, 7: 586, 8: 501, 2: 474}
   benign set to: 0
uav_attack label counts: {2: 8109, 0: 1460, 1: 498}
   benign set to: 2


In [31]:
# Train all 4 models on uav_attack (smallest dataset, fastest to test).
# AE and Isolation Forest take the dataset name so they use the correct
# benign label for training on normal traffic only.

import models
importlib.reload(models)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
rf  = models.build_random_forest(Xtr, ytr)
cnn = models.build_cnn(Xtr, ytr)
ae  = models.build_autoencoder(Xtr, ytr, "uav_attack")
iso = models.build_isolation_forest(Xtr, ytr, "uav_attack")
print("all four trained")

Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 118ms/step - accuracy: 0.9173 - loss: 0.2686 - val_accuracy: 1.0000 - val_loss: 8.9431e-04
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - accuracy: 0.9988 - loss: 0.0048 - val_accuracy: 0.9988 - val_loss: 0.0010
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.9995 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 1.4127e-04
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - accuracy: 0.9994 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 3.1931e-04
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 142ms/step - accuracy: 0.9996 - loss: 0.0022 - val_accuracy: 1.0000 - val_loss: 1.7292e-04
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.4303 - val_loss: 0.3882
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3850 - val_loss: 0.3451
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3381 - val_loss: 0.2792
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2627 - val_loss: 0.2131
Epoch 5/10


In [9]:
# load_dataset reads the CSV, drops bad rows, separates features from the
# label, scales everything, and splits 70/30. It hands back 5 things:
# training and test features, training and test labels, and feature names.

import importlib, config, data_loader, models, metrics             # import the 4 modules
for m in(config, data_loader, models, metrics): importlib.reload(m) # forcing python to re-read them from disk

# Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack") -> below is for this line function:
# load_dataset function for eading the CSV drops bad rows, seperates features from the label, scales everthing, and splits 70/30. 
# it hands back 5 things: training and test features, traiing and test labels and feature names
Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
rf = models.build_random_forest(Xtr, ytr)                   # RF and cnn only need features and labels (supervised)
cnn = models.build_cnn(Xtr, ytr)
ae = models.build_autoencoder(Xtr, ytr,"uav_attack")       # AE and Isolation Forest are unsupervised, both train on normal traffic only
iso = models.build_isolation_forest(Xtr, ytr, "uav_attack")

# below is evaluate and print above out:
"""
evaluate_supervised -> RF, CNN = both predict classes — same logic, one flag for CNN's reshape
evaluate_autoencoder -> AE = needs MSE + threshold conversion
evaluate_isolation_forest -> IF = needs +1/−1 conversion
"""
print("RF :", metrics.evaluate_supervised(rf, Xte, yte))                      # rf and cnn are share same function 
print("CNN:", metrics.evaluate_supervised(cnn, Xte, yte, is_cnn=True))        # only cnn needs are reshape and an argmax which the is_cnn=True flag handles
print("AE :", metrics.evaluate_autoencoder(ae, Xte, yte, "uav_attack"))       # ae outputs a reconstruction error, iso outputs +1 or -1 directly (inlier/outlier)
print("IF :", metrics.evaluate_isolation_forest(iso, Xte, yte, "uav_attack")) # ae and iso is not share a function



Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.8435 - loss: 0.3313 - val_accuracy: 0.9988 - val_loss: 0.0018
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.9981 - loss: 0.0075 - val_accuracy: 1.0000 - val_loss: 2.8796e-04
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.9985 - loss: 0.0069 - val_accuracy: 1.0000 - val_loss: 3.3805e-04
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.9993 - loss: 0.0022 - val_accuracy: 1.0000 - val_loss: 4.5922e-04
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - accuracy: 0.9994 - loss: 0.0028 - val_accuracy: 1.0000 - val_loss: 3.2157e-04
Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.4490 - val_loss: 0.4010
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.4000 - val_loss: 0.3593
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3499 - val_loss: 0.2967
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2855 - val_loss: 0.2307
Epoch 5/10

In [74]:
# Test SHAP on Random Forest. Returns (importance, all_zero) after the
# return-type standardization. top-5 by absolute SHAP magnitude.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
rf = models.build_random_forest(Xtr, ytr)

X_sample = Xte[:config.SHAP_SAMPLES_RF]
imp, all_zero = xai.shap_random_forest(rf, X_sample, feat) # two values

print("all_zero:", all_zero, "| shape:", imp.shape)
top5 = np.argsort(imp)[::-1][:5]
print("top 5 features:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

all_zero: False | shape: (80,)
top 5 features:
  eph_y: 0.024562
  noise_per_ms: 0.020652
  epv_y: 0.016969
  s_variance_m_s: 0.016265
  evh: 0.014976


In [77]:
# Test SHAP on CNN (GradientExplainer, needs 3D input + background data).
# Returns (importance, all_zero). Retrains CNN each run (~20s).
# The Keras UserWarning about input structure is harmless (Paper 1 showed it too).

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
cnn = models.build_cnn(Xtr, ytr)

X_sample = Xte[:config.SHAP_SAMPLES_CNN]    # 100 samples
X_bg     = Xtr[:config.SHAP_BACKGROUND]     # 50 background samples

imp,all_zero = xai.shap_cnn(cnn, X_sample, X_bg, feat)

print("shape:", imp.shape, "(should be", len(feat), "features)")
top5 = np.argsort(imp)[::-1][:5]
print("top 5 features:")
for i in top5:
    print(f" {feat[i]}: {imp[i]:.6f}")

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 167ms/step - accuracy: 0.8615 - loss: 0.3417 - val_accuracy: 0.9957 - val_loss: 0.0091
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step - accuracy: 0.9977 - loss: 0.0088 - val_accuracy: 0.9957 - val_loss: 0.0084
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 148ms/step - accuracy: 0.9981 - loss: 0.0066 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.9995 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.9993 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 6.0334e-04


/opt/miniconda3/envs/paper2_env/lib/python3.9/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_115
Received: inputs=['Tensor(shape=(100, 80, 1))']
  warnings.warn(msg)
/opt/miniconda3/envs/paper2_env/lib/python3.9/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_115
Received: inputs=['Tensor(shape=(50, 80, 1))']
  warnings.warn(msg)


shape: (80,) (should be 80 features)
top 5 features:
 noise_per_ms: 0.022641
 epv_y: 0.021952
 epv: 0.013757
 evh: 0.013540
 eph: 0.012921


In [78]:
# Test SHAP on Autoencoder (KernelExplainer via reconstruction-error wrapper).
# SLOWEST method: ~5 min on 50 samples, because KernelExplainer runs the
# AE thousands of times. Returns (importance, all_zero).
# Explains only 50 samples (SHAP_SAMPLES_AE) - the full test set would take hours.

import numpy as np # for mathematical calculations, for sortting features importance scores
# importlib for reload a modules.
import importlib, config, data_loader, models, metrics, xai

# reload every new modules, otherwise jupyter may still run old code that from paper 1
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

# loading the dataset.
# Xtr = training features, Xte = test features, ytr = training labels, yte = test labels, feat = feature names
Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")

# ae learning only normal traffic
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")

X_sample = Xte[:config.SHAP_SAMPLES_AE]  # shap is expensive, we test dataset 15000, but explain only first 50 samples
X_bg = Xtr[:config.SHAP_BACKGROUND]     # 50 background samples

imp, all_zero = xai.shap_autoencoder(ae, X_sample, X_bg, feat) # run shap and ask: which features were most important?

print("shape:", imp.shape,"(should be", len(feat), "features)") # check result: imp.shape shows how many importance scores we received
top5 = np.argsort(imp)[::-1][:5] # show results and sort it from smallest -> largest
print("top 5 features: ")
for i in top5:
    print(f" {feat[i]}: {imp[i]:.6f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.4463 - val_loss: 0.3947
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.4053 - val_loss: 0.3711
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.3817 - val_loss: 0.3311
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3374 - val_loss: 0.2758
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2748 - val_loss: 0.2146
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.2184 - val_loss: 0.1665
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1732 - val_loss: 0.1401
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1486 - val_loss: 0.1260
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1306 - val_loss: 0.1173
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.1205 - val_loss: 0.1106


  0%|          | 0/50 [00:00<?, ?it/s]

shape: (80,) (should be 80 features)
top 5 features: 
 xy_valid: 0.084000
 v_xy_valid: 0.079071
 q[3]: 0.072979
 vdop: 0.062434
 q[1]: 0.060811


In [79]:
# Test SHAP on Isolation Forest (KernelExplainer via anomaly-score wrapper).
# Faster than AE SHAP (~40s) because tree scoring is cheap. Returns (importance, all_zero).

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
iso = models.build_isolation_forest(Xtr, ytr, "uav_attack")

X_sample = Xte[:config.SHAP_SAMPLES_AE]  #  50 samples (KernelExplainer is slow)
X_bg = Xtr[:config.SHAP_BACKGROUND]      # 50 background samples

imp,all_zero = xai.shap_isolation_forest(iso, X_sample, X_bg, feat)

print("shape:", imp.shape, "(should be", len(feat), "features)")
top5 = np.argsort(imp)[::-1][:5]
print("top 5 features:")
for i in top5:
    print(f" {feat[i]}: {imp[i]:.6f}")

  0%|          | 0/50 [00:00<?, ?it/s]

shape: (80,) (should be 80 features)
top 5 features:
 s_variance_m_s: 0.010221
 evh: 0.007638
 vdop: 0.006385
 eph: 0.006241
 vel_n_m_s: 0.005965


In [80]:
# Test PI on Random Forest. Expect all_zero=True: RF hits 100% F1, so
# shuffling any single feature doesn't drop the score -> PI measures nothing.
# This is a documented finding: PI is uninformative for models at ceiling.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)
Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
rf = models.build_random_forest(Xtr, ytr)

# expect: all_zero = true (RF is at 100# f1, paper 1 saw the same)
imp, all_zero = xai.pi_random_forest(rf, Xte, yte, feat, n_repeats = 10)
print("all zero:", all_zero)
print("max importance:", imp.max())
print("non-zero features:", np.sum(imp !=0))

all zero: True
max importance: 0.0
non-zero features: 0


In [81]:
# Test PI on CNN (manual implementation - sklearn can't handle Keras).
# WARNING: slowest method, ~8 min at n_repeats=10.
# Note: CNN hit 100% F1, so PI returned all_zero=True here too - same as RF.
# (Paper 1's CNN was 99.97% and got small non-zero values; our 100% gives zeros.)

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
cnn = models.build_cnn(Xtr, ytr)

# expect: all_zero = False, small non-zero values (~0.001-0.01)
# Paper 1 top features: vel_m_s, vel_n_m_s, hdop, evv, jamming_indicator
imp, all_zero = xai.pi_cnn(cnn, Xte, yte, feat, n_repeats=10)

print("all_zero:", all_zero)
print("max importance:", imp.max())
print("non-zero features:", np.sum(imp != 0))
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 104ms/step - accuracy: 0.8072 - loss: 0.4081 - val_accuracy: 0.9957 - val_loss: 0.0126
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.9995 - loss: 0.0058 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 96ms/step - accuracy: 0.9990 - loss: 0.0049 - val_accuracy: 0.9986 - val_loss: 0.0019
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step - accuracy: 0.9992 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 9.4144e-04
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.9994 - loss: 0.0034 - val_accuracy: 1.0000 - val_loss: 6.2856e-04
all_zero: False
max importance: 0.004295851615030466
non-zero features: 37
top 5:
  vel_m_s: 0.004296
  hdop: 0.003916
  vel_n_m_s: 0.003468
  vdop: 0.001618
  q[2]: 0.001347


In [82]:
# Test PI on Autoencoder (~3 min). The AE is at ~90% F1, well below ceiling,
# so PI IS informative here - unlike RF. Converts reconstruction error to a
# binary prediction via the 95th-percentile threshold, then shuffles features.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")

# expect: all_zero = False (AE is at 90% F1, so there IS room to drop)
# Paper 1 top features: lat_x, lat_y, x, q[3], q[2]  (position coordinates)
imp, all_zero = xai.pi_autoencoder(ae, Xte, yte, feat, "uav_attack", n_repeats=10)

print("all_zero:", all_zero)
print("max importance:", imp.max())
print("non-zero features:", np.sum(imp != 0))
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.4315 - val_loss: 0.3783
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.3931 - val_loss: 0.3404
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3516 - val_loss: 0.2861
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2921 - val_loss: 0.2250
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.2288 - val_loss: 0.1746
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1859 - val_loss: 0.1494
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1609 - val_loss: 0.1359
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1467 - val_loss: 0.1271
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1380 - val_loss: 0.1203
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1266 - val_loss: 0.1144
all_zero: False
max importance: 0.06808592381770637
non-zero features: 64
top 5:
  dist_bottom_valid: 0.068086
  terrain_alt_valid: 0.065986
  terrai

In [84]:
# ae n-repeats = 10
# Sensitivity check: AE PI at n_repeats=5 vs the 10-repeat run above.
# If the top-5 is similar, 5 repeats is enough - justifies the faster setting.
# NOTE: this cell retrains the AE, so results mix "fewer repeats" with a new
# model. For a clean comparison, reuse the same ae from the n_repeats=10 cell.
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")

# expect: all_zero = False (AE is at 90% F1, so there IS room to drop)
# Paper 1 top features: lat_x, lat_y, x, q[3], q[2]  (position coordinates)
imp, all_zero = xai.pi_autoencoder(ae, Xte, yte, feat, "uav_attack", n_repeats=10)

print("all_zero:", all_zero)
print("max importance:", imp.max())
print("non-zero features:", np.sum(imp != 0))
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.4345 - val_loss: 0.3835
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.3986 - val_loss: 0.3492
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.3610 - val_loss: 0.2983
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3054 - val_loss: 0.2379
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2480 - val_loss: 0.1892
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1955 - val_loss: 0.1559
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1676 - val_loss: 0.1361
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1446 - val_loss: 0.1235
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1298 - val_loss: 0.1141
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1186 - val_loss: 0.1064
all_zero: False
max importance: 0.06709702563662348
non-zero features: 64
top 5:
  dist_bottom_valid: 0.067097
  terrain_alt: 0.064037
  terrain_alt_

In [83]:
# ae n-repeats = 5
# Sensitivity check: AE PI at n_repeats=5 vs the 10-repeat run above.
# If the top-5 is similar, 5 repeats is enough - justifies the faster setting.
# NOTE: this cell retrains the AE, so results mix "fewer repeats" with a new
# model. For a clean comparison, reuse the same ae from the n_repeats=10 cell.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")

# expect: all_zero = False (AE is at 90% F1, so there IS room to drop)
# Paper 1 top features: lat_x, lat_y, x, q[3], q[2]  (position coordinates)
imp5, all_zero5 = xai.pi_autoencoder(ae, Xte, yte, feat, "uav_attack", n_repeats=5)

print("all_zero:", all_zero5)
print("max importance:", imp5.max())
print("non-zero features:", np.sum(imp5 != 0))
top5_new = np.argsort(imp5)[::-1][:5]
print("n_repeats=5 top 5:")
print("top 5:")
for i in top5_new:
    print(f"  {feat[i]}: {imp5[i]:.6f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.4458 - val_loss: 0.3947
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.4104 - val_loss: 0.3659
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3741 - val_loss: 0.3208
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3199 - val_loss: 0.2500
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2496 - val_loss: 0.1848
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1882 - val_loss: 0.1497
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1569 - val_loss: 0.1305
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1382 - val_loss: 0.1203
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1246 - val_loss: 0.1132
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1162 - val_loss: 0.1074
all_zero: False
max importance: 0.0674966697143835
non-zero features: 64
n_repeats=5 top 5:
top 5:
  terrain_alt: 0.067497
  dist_bottom_valid: 0.065

In [85]:
# Test PI on Isolation Forest. Returns (importance, all_zero).
# NOTE: earlier this returned all-zeros due to a bug (predicted on X_sample
# instead of the shuffled X_perm). Fixed -> now 60 non-zero features.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
iso = models.build_isolation_forest(Xtr, ytr, "uav_attack")

# expect: all_zero = False (IF is at 79% F1, plenty of room to drop)
# no Paper 1 baseline - new model
imp, all_zero = xai.pi_isolation_forest(iso, Xte, yte, feat, "uav_attack", n_repeats=10)

print("all_zero:", all_zero)
print("max importance:", imp.max())
print("non-zero features:", np.sum(imp != 0))
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

all_zero: False
max importance: 0.025881407015146896
non-zero features: 60
top 5:
  evv: 0.025881
  delta_q_reset[1]: 0.025740
  q[3]: 0.024076
  heading_y: 0.021745
  eph_x: 0.019490


In [86]:
# Test LIME on Random Forest (default n_samples=50 now).
# Returns (importance, all_zero) - standardized like the other methods.
# LIME picks position/velocity features (lat_y, satellites_used) - notably
# different from SHAP's GPS-error features. That SHAP-vs-LIME divergence is
# a key thing ESS will measure.
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
rf = models.build_random_forest(Xtr, ytr)

imp, all_zero = xai.lime_random_forest(rf, Xtr, Xte, feat)   # two values now
print("all_zero:", all_zero, "| shape:", imp.shape)
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

all_zero: False | shape: (80,)
top 5:
  lat_y: 0.064372
  satellites_used: 0.026687
  vel_n_m_s: 0.025474
  hdop: 0.022868
  lat_x: 0.018657


In [87]:
# Test LIME on CNN (wrapper reshapes 2D->3D). Returns (importance, all_zero).
# CNN LIME favours terrain/position features, diverging from CNN SHAP's
# GPS-error features - the SHAP-vs-LIME split shows up on the CNN too.
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
cnn = models.build_cnn(Xtr, ytr)

imp, all_zero = xai.lime_cnn(cnn, Xtr, Xte, feat)   # two values, default n_samples=50
print("all_zero:", all_zero, "| shape:", imp.shape)
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - accuracy: 0.8914 - loss: 0.3510 - val_accuracy: 0.9972 - val_loss: 0.0065
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.9992 - loss: 0.0086 - val_accuracy: 0.9972 - val_loss: 0.0045
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.9993 - loss: 0.0036 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step - accuracy: 0.9993 - loss: 0.0024 - val_accuracy: 1.0000 - val_loss: 9.6525e-04
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.9990 - loss: 0.0030 - val_accuracy: 1.0000 - val_loss: 9.3479e-04
all_zero: False | shape: (80,)
top 5:
  terrain_alt: 0.145004
  dist_bottom_valid: 0.137248
  vel_m_s: 0.107022
  lat_y: 0.095870
  satellites_used: 0.082889


In [88]:
# Test LIME on Autoencoder (needs threshold to convert MSE->probability).
# Returns (importance, all_zero). AE LIME tends to agree more with AE PI
# (both perturbation-based) than with AE SHAP.
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")   # train it in this cell

ae_res = metrics.evaluate_autoencoder(ae, Xte, yte, "uav_attack")
threshold = ae_res["threshold"]
print("threshold:", threshold)

imp, all_zero = xai.lime_autoencoder(ae, Xtr, Xte, feat, threshold)   # two values, default 50
print("all_zero:", all_zero, "| shape:", imp.shape)
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.4442 - val_loss: 0.3875
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.3994 - val_loss: 0.3476
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3528 - val_loss: 0.2902
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.2965 - val_loss: 0.2246
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2267 - val_loss: 0.1736
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.1817 - val_loss: 0.1438
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.1518 - val_loss: 0.1288
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1384 - val_loss: 0.1198
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1261 - val_loss: 0.1126
Epoch 10/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1160 - val_loss: 0.1065
threshold: 0.28545385704715903
all_zero: False | shape: (80,)
top 5:
  dist_bottom_valid: 0.021078
  terrain_alt: 0.019731
  terrain_alt_valid: 0.019

In [89]:
# Test LIME on Isolation Forest. Returns (importance, all_zero).
# NOTE: IF PI is NOT all-zeros anymore (that was the X_sample/X_perm bug).
# So IF has 3 usable methods (SHAP, PI, LIME), not 2. Update earlier notes.
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
iso = models.build_isolation_forest(Xtr, ytr, "uav_attack")

imp, all_zero = xai.lime_isolation_forest(iso, Xtr, Xte, feat)   # two values, default 50
print("all_zero:", all_zero, "| shape:", imp.shape)
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

all_zero: False | shape: (80,)
top 5:
  terrain_alt_valid: 0.013702
  dist_bottom_valid: 0.012526
  terrain_alt: 0.010211
  vdop: 0.008828
  s_variance_m_s: 0.007815


In [90]:
# Test IG on CNN (gradient-based, neural only). Returns (importance, all_zero).
# Fixed-class version. IG is gradient-based like SHAP - check if it lands
# nearer SHAP (GPS-error features) than LIME (terrain features).
import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)

Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
cnn = models.build_cnn(Xtr, ytr)

imp, all_zero = xai.ig_cnn(cnn, Xte[:100], feat, n_steps=50)
print("all_zero:", all_zero, "| max importance:", imp.max())
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f"  {feat[i]}: {imp[i]:.6f}")

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.8362 - loss: 0.3840 - val_accuracy: 0.9957 - val_loss: 0.0137
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.9985 - loss: 0.0089 - val_accuracy: 0.9972 - val_loss: 0.0061
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.9987 - loss: 0.0077 - val_accuracy: 0.9972 - val_loss: 0.0063
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.9988 - loss: 0.0035 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.9988 - loss: 0.0047 - val_accuracy: 1.0000 - val_loss: 8.4129e-04
all_zero: False | max importance: 0.0340723773278296
top 5:
  noise_per_ms: 0.034072
  epv_y: 0.029828
  satellites_used: 0.029670
  jamming_indicator: 0.028558
  eph: 0.023133


In [67]:
# Test IG on Autoencoder (gradient-based, attributes reconstruction-error MSE).
# Returns (importance, all_zero). Fast - no KernelExplainer.
# Two-camps check on AE:
#   AE SHAP: xy_valid, v_xy_valid, noise_per_ms, epv_y, y      (gradient camp)
#   AE PI:   dist_bottom_valid, terrain_alt_valid, terrain_alt, lat_y, x  (perturbation)
#   AE LIME: dist_bottom_valid, terrain_alt, terrain_alt_valid, noise_per_ms, q[2]  (perturbation)
# IG is gradient-based -> expect it near SHAP, far from PI/LIME.

import numpy as np
import importlib, config, data_loader, models, metrics, xai
for m in (config, data_loader, models, metrics, xai): importlib.reload(m)
Xtr, Xte, ytr, yte, feat = data_loader.load_dataset("uav_attack")
ae = models.build_autoencoder(Xtr, ytr, "uav_attack")

# AE SHAP: xy_valid, v_xy_valid, noise_per_ms, epv_y, y
# AE PIL dist_bottom_valid, terrain_alt_valid, terrain_alt, eph, lat_y
# AE LIME: terrain_alt, terrain_alt_valid, dist_bottom_valid, vel_m_s, noise_per_ms
# IG is gradient-based - expect it near SHAP, far from PI/LIME (the perturbation camp)
imp, all_zero = xai.ig_autoencoder(ae, Xte[:100], feat, n_steps=50)

print("all_zero:", all_zero)
print("max importance:", imp.max())
top5 = np.argsort(imp)[::-1][:5]
print("top 5:")
for i in top5:
    print(f" {feat[i]}: {imp[i]:.6f}")

Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.4395 - val_loss: 0.3985
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4010 - val_loss: 0.3680
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3665 - val_loss: 0.3238
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3146 - val_loss: 0.2668
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2547 - val_loss: 0.2094
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2007 - val_loss: 0.1638
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1621 - val_loss: 0.1345
Epoch 8/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1321 - val_loss: 0.1178
Epoch 9/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1185 - val_loss: 0.1081
Epoch 10/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1091 - val_loss: 0.1014
all_zero: False
max importance: 0.03537055524005325
top 5:
 xy_valid: 0.035371
 v_xy_valid: 0.032887
 epv_y: 0.020338
 noise_per_ms: 0.017787
 eph_y: 